# Chapter 11 — ML 기반 이상 탐지 (실습)

## 실습 목표
1. BENIGN 만으로 K-Means / DBSCAN / Isolation Forest 학습
2. 전체에 예측 적용
3. ROC-AUC, P/R 비교
4. contamination 튜닝

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q pandas
# !pip install -q numpy
# !pip install -q matplotlib
# !pip install -q scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
rng = np.random.default_rng(42)

In [ ]:
# 데이터 — ch09 형식
def make_rows(n, label, dur, pkts, syn):
    return pd.DataFrame({
        'Flow Duration': rng.uniform(*dur, size=n),
        'Total Fwd Packets': rng.integers(*pkts, size=n),
        'SYN Flag Count': rng.uniform(*syn, size=n),
        'Label': label,
    })
df = pd.concat([
    make_rows(16000, 'BENIGN',   (1000, 1_000_000), (5, 100),    (0, 0.05)),
    make_rows( 2300, 'DDoS',     (10, 5000),         (500, 5000), (0.4, 0.7)),
    make_rows( 1200, 'PortScan', (1, 200),           (1, 5),      (0.85, 1.0)),
], ignore_index=True).sample(frac=1, random_state=0).reset_index(drop=True)
df['log_dur'] = np.log1p(df['Flow Duration'])
df['log_pkts'] = np.log1p(df['Total Fwd Packets'])
FEATURES = ['log_dur', 'log_pkts', 'SYN Flag Count']
y = (df['Label'] != 'BENIGN').astype(int)

scaler = RobustScaler()
from sklearn.model_selection import train_test_split
train, held_out = train_test_split(df, test_size=0.3, stratify=y, random_state=42)
fit_train, validation = train_test_split(train, test_size=0.2, stratify=train['Label'], random_state=43)
normal_train = fit_train[fit_train['Label']=='BENIGN']
normal_valid = validation[validation['Label']=='BENIGN']
X_normal = scaler.fit_transform(normal_train[FEATURES])
X_valid_normal = scaler.transform(normal_valid[FEATURES])
X_valid = scaler.transform(validation[FEATURES])
y_valid = (validation["Label"] != "BENIGN").astype(int)
X = scaler.transform(held_out[FEATURES])
y = (held_out['Label'] != 'BENIGN').astype(int)
print(f'X.shape={X.shape}, normal={len(X_normal)}, anomaly_rate={y.mean():.2%}')

---
## Step 1. K-Means

In [ ]:
# ── 실습 과제 ──
# 목표: Step 1. K-Means
# 만들 결과: km, scores_km, auc_km
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
km = ...
scores_km = ...
auc_km = ...
...
```
</details>


---
## Step 2. Isolation Forest

In [ ]:
# ── 실습 과제 ──
# 목표: Step 2. Isolation Forest
# 만들 결과: iforest, scores_if, auc_if
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
iforest = ...
scores_if = ...
auc_if = ...
print(f'Isolation Forest AUC = ...
```
</details>


---
## Step 3. DBSCAN — 일부 샘플만 (느림)

In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. DBSCAN — 일부 샘플만 (느림)
# 만들 결과: idx, X_small, db, pred_db, auc_db
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
idx = ...
X_small = ...
db = ...
pred_db = ...
auc_db = ...
print(f'DBSCAN 배치 내 잡음 지표 AUC = ...
```
</details>


---
## Step 4. 세 모델 비교

In [ ]:
comparison = pd.DataFrame([
    {'model': 'K-Means',          'AUC': round(auc_km, 3)},
    {'model': 'Isolation Forest', 'AUC': round(auc_if, 3)},
])
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(comparison['model'], comparison['AUC'],
        color=['#1a5f7a', '#e74c3c'])
for i, v in enumerate(comparison['AUC']):
    ax.text(v + 0.005, i, f'{v}', va='center')
ax.set_xlim(0.5, 1.0)
ax.set_xlabel('ROC-AUC'); ax.set_title('Anomaly Detection Models')
plt.tight_layout(); plt.show()
comparison

---
## AutoEncoder 기반 이상 탐지 (딥러닝 확장)

신경망으로 **정상 트래픽을 압축·복원** 하도록 학습한 뒤 복원 오차가 큰 샘플을 이상으로 본다.
입력을 비선형으로 압축·복원하는 모델입니다. TensorFlow가 필요하며, 실행 환경의 설치 여부를 확인합니다.
- 목표: 정상(X_normal)으로 AutoEncoder 학습 → 복원 오차를 이상 점수로 → AUC 측정


In [ ]:
# ── 실습 과제 ──
# 목표: AutoEncoder 기반 이상 탐지 (딥러닝 확장) — 신경망으로 **정상 트래픽을 압축·복원** 하도록 학습한 뒤 복원 오차가 큰 샘플을 이상으로 본다.
# 만들 결과: d, ae, mse_norm, threshold, scores_ae, auc_ae
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.


<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# AutoEncoder 이상 탐지 — tensorflow 필요(Colab 기본 포함, 로컬은 pip install tensorflow)
try:
    from tensorflow import keras
    from tensorflow.keras import layers
    d = ...
    ae = ...
        layers.Input(shape = ...
        layers.Dense(8, activation = ...
        layers.Dense(2, activation = ...
        layers.Dense(8, activation = ...
        layers.Dense(d, activation = ...
    ...
    ae.compile(optimizer = ...
    ae.fit(X_normal, X_normal, epochs = ...

    mse_norm = ...
    threshold = ...
    scores_ae = ...
    auc_ae = ...
    print(f'AutoEncoder AUC = ...
except ImportError:
    ...
```
</details>


---
## Step 5. Isolation Forest — Threshold 튜닝

In [ ]:
# ── 실습 과제 ──
# 목표: Step 5. Isolation Forest — Threshold 튜닝
# 만들 결과: valid_scores, thresholds, rows, pred, tune, best
# 산출: 그래프 시각화
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
valid_scores = ...
thresholds = ...
from sklearn.metrics import precision_score, recall_score, f1_score
rows = ...
for t in thresholds:
    pred = ...
    ...
                 'precision': precision_score(y_valid, pred, zero_division = ...
                 'recall': recall_score(y_valid, pred, zero_division = ...
                 'f1': f1_score(y_valid, pred, zero_division = ...
tune = ...
best = ...
print(f'Best F1 = ...
print(f'  Precision = ...

fig, ax = ...
ax.plot(tune['threshold'], tune['precision'], label = ...
ax.plot(tune['threshold'], tune['recall'],    label = ...
ax.plot(tune['threshold'], tune['f1'],        label = ...
ax.axvline(best['threshold'], color = ...
...
final_pred = ...
...
```
</details>


---
## 정리

- Isolation Forest·DBSCAN 등의 성능과 실행 시간은 데이터 분포·크기·설정에 따라 비교한다.
- DBSCAN 은 작은 데이터에 빠르지만 대용량은 어렵다.
- Threshold 튜닝으로 운영 시나리오(high recall vs high precision)에 맞춤.
- 다음 Part의 12~13장에서 로그·상관분석을 다루고, 14장에서 지도학습으로 넘어간다.